# 🎯 UML Generator - gpt-oss-20b (Colab Server)

Notebook này:
1. Cài đặt thư viện cần thiết
2. Tải model `gpt-oss-20b-UML-Generator` (IQ4_XS ~12GB) từ HuggingFace
3. Chạy FastAPI server để nhận request sinh UML
4. Expose ra internet qua **ngrok**

**⚠️ Yêu cầu:**
- Bật GPU: `Runtime > Change runtime type > T4 GPU`
- Có tài khoản ngrok (miễn phí): https://ngrok.com/signup


## Bước 1: Cài đặt thư viện

In [1]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
  --quiet 
# Cài các thư viện khác
!pip install fastapi uvicorn pyngrok huggingface_hub --quiet

print("✅ Cài đặt hoàn tất!")

## Bước 2: Tải model từ HuggingFace

> ⏳ **Lần đầu mất ~5-10 phút** để tải ~12GB. Các lần sau nếu dùng Colab Pro có thể mount Google Drive để cache.

In [ ]:
from huggingface_hub import hf_hub_download
import os

print("⬇️  Đang tải model gpt-oss-20b-UML-Generator (IQ4_XS ~12GB)...")
print("   Vui lòng đợi, quá trình này mất vài phút.")

model_path = hf_hub_download(
    repo_id="mradermacher/gpt-oss-20b-UML-Generator-GGUF",
    filename="gpt-oss-20b-UML-Generator.IQ4_XS.gguf",
    
)

print(f"\n✅ Model đã tải xong: {model_path}")
print(f"   File size: {os.path.getsize(model_path) / 1024**3:.1f} GB")

## Bước 3: Load model vào GPU

In [ ]:
from llama_cpp import Llama

print("🔄 Đang load model vào GPU...")

llm = Llama(
    model_path=model_path,
    n_ctx=4096,        # Context window - giữ 4096 cho Free Tier T4
    n_gpu_layers=-1,   # Offload TẤT CẢ layers lên GPU
    n_threads=2,       # CPU threads
    verbose=False,
)

print("✅ Model đã load xong và sẵn sàng!")

# Test nhanh
test = llm("Generate a simple UML:", max_tokens=20)
print(f"   Test output: {test['choices'][0]['text'][:50]}...")

## Bước 4: Khởi động FastAPI Server

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading
import time

app = FastAPI(title="UML Generator - gpt-oss-20b API")

# Cho phép CORS (để LLM Service ở máy bạn gọi được)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 2048
    temperature: float = 0.1
    stop: list = ["</s>", "<|end|>", "<|im_end|>"]

@app.get("/health")
def health():
    return {
        "status": "online",
        "model": "gpt-oss-20b-UML-Generator-IQ4_XS",
        "context_size": 4096,
    }

@app.post("/generate")
async def generate(req: GenerateRequest):
    if not req.prompt.strip():
        raise HTTPException(status_code=400, detail="Prompt không được để trống")

    try:
        start = time.time()
        output = llm(
            req.prompt,
            max_tokens=req.max_tokens,
            temperature=req.temperature,
            stop=req.stop,
        )
        latency = round(time.time() - start, 2)
        text = output["choices"][0]["text"].strip()

        return {
            "text": text,
            "latency_seconds": latency,
            "usage": output.get("usage", {}),
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Chạy server trong background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)  # Đợi server khởi động

print("✅ FastAPI server đang chạy trên port 8001")

## Bước 5: Expose qua ngrok

> 🔑 **Lấy Auth Token:** Đăng nhập https://ngrok.com → Dashboard → "Your Authtoken" → Copy

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
# ⚠️ THAY TOKEN CỦA BẠN VÀO ĐÂY
NGROK_AUTH_TOKEN = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)   

# Tạo tunnel từ port 8001 ra internet
public_url = ngrok.connect(8001)

print("=" * 60)
print("🚀 SERVER SẴN SÀNG!")
print("=" * 60)
print(f"\n📌 URL NGROK CỦA BẠN:")
print(f"   {public_url}")
print(f"\n📋 COPY DÒNG NÀY VÀO FILE .env TRÊN MÁY BẠN:")
print(f"   COLAB_API_URL={public_url}")
print()
print(f"🔗 Health check: {public_url}/health")
print("=" * 60)
print("⚠️  Đừng đóng tab này! Server sẽ dừng nếu bạn đóng.")

## Bước 6: Keep-alive (tùy chọn)

Chạy cell này để tránh Colab tự ngắt kết nối sau 90 phút idle.

In [ ]:
import time
import requests

print("🔄 Keep-alive đang chạy... (Ctrl+C để dừng)")
count = 0
while True:
    try:
        r = requests.get("http://localhost:8001/health", timeout=5)
        count += 1
        print(f"   [{count}] Server OK - {time.strftime('%H:%M:%S')}")
    except Exception as e:
        print(f"   ⚠️  Server lỗi: {e}")
    time.sleep(60)  # Ping mỗi 60 giây